# Data quality reporting
- investigating why so few files produced valid .npy files

In [ ]:
# gemini wrote me this function for scanning the jsons
import json
from pathlib import Path
from collections import Counter
import pandas as pd

def analyze_json_reports(base_dir="sensori-data"):
    base_path = Path(base_dir)
    json_files = list(base_path.glob("*/info.json"))

    if not json_files:
        print(f"No info.json files found in '{base_dir}' directory.")
        return

    total_samples = len(json_files)
    eligible_count = 0
    ineligible_count = 0

    # Counters for diagnostics
    failed_criteria_counter = Counter()
    eligibility_checks_counter = Counter()
    missing_keys_count = 0

    sample_summary = []

    for file_path in json_files:
        sample_id = file_path.parent.name

        try:
            with open(file_path, "r") as f:
                data = json.load(f)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        # Extract eligibility and failed criteria
        is_eligible = data.get("participant_eligibility", 0) == 1
        failed_criteria = data.get("failed_criteria", [])
        eligibility_dict = data.get("eligibility", {})

        if is_eligible:
            eligible_count += 1
        else:
            ineligible_count += 1

        # Track explicitly listed failed criteria
        if failed_criteria:
            for criterion in failed_criteria:
                failed_criteria_counter[criterion] += 1

        # Track individual eligibility flags that evaluated to False
        for key, status in eligibility_dict.items():
            if not status:
                eligibility_checks_counter[key] += 1

        # Record detailed entry
        sample_summary.append({
            "sample_id": sample_id,
            "eligible": is_eligible,
            "failed_criteria": ", ".join(failed_criteria) if failed_criteria else "None",
            "eligible_days": data.get("colap_parser", {}).get("eligible_days", None),
            "sample_rate_hz": data.get("SampleRate", None),
            "wear_time_days": data.get("nonwear_detection", {}).get("WearTime(days)", None)
        })

    # Summary Report
    print("=" * 60)
    print("                SENSOR DATA PIPELINE REPORT                ")
    print("=" * 60)
    print(f"Total Samples Analyzed: {total_samples}")
    print(f"Passed / Eligible:      {eligible_count} ({eligible_count / total_samples * 100:.1f}%)")
    print(f"Failed / Ineligible:    {ineligible_count} ({ineligible_count / total_samples * 100:.1f}%)")
    print("-" * 60)

    if failed_criteria_counter:
        print("\n[Breakdown of 'failed_criteria' entries]:")
        for criterion, count in failed_criteria_counter.most_common():
            pct = (count / total_samples) * 100
            print(f"  - {criterion:<30}: {count} ({pct:.1f}%)")

    if eligibility_checks_counter:
        print("\n[Breakdown of Failed Eligibility Checks]:")
        for check_name, count in eligibility_checks_counter.most_common():
            pct = (count / total_samples) * 100
            print(f"  - {check_name:<30}: {count} ({pct:.1f}%)")

    # Save detailed summary to CSV for deeper exploration
    df = pd.DataFrame(sample_summary)
    output_csv = "pipeline_data_quality_report.csv"
    df.to_csv(output_csv, index=False)
    print("\n" + "=" * 60)
    print(f"Detailed per-sample report saved to: {output_csv}")
    print("=" * 60)

In [1]:
import json
from pathlib import Path
import os
from collections import Counter

failed_criteria_counts = Counter()
eligibility_counts = Counter()
total, eligible = 0, 0

for json_path in Path("../data/sensori-data").rglob("info.json"):
    data = json.loads(json_path.read_text())
    total += 1

    if data.get("participant_eligibility") == 1:
        eligible += 1

    failed_criteria_counts.update(data.get("failed_criteria", []))
    eligibility_counts.update(
        k for k, v in data.get("eligibility", {}).items() if not v
    )

print(f"Total: {total} | Eligible: {eligible} ({eligible/total*100:.1f}%) | Ineligible: {total - eligible}")
print("\nFailed Criteria:", dict(failed_criteria_counts))
print("Failed Eligibility Checks:", dict(eligibility_counts))

Total: 2403 | Eligible: 1033 (43.0%) | Ineligible: 1370

Failed Criteria: {'calibration_ok': 1314, 'eligible_day_available': 127, 'mean_acceleration_ok': 12, 'interrupts_ok': 11}
Failed Eligibility Checks: {'calibration_ok': 1314, 'eligible_day_available': 127, 'mean_acceleration_ok': 12, 'interrupts_ok': 11}


In [4]:
# gem gave me this too to check sites and sensors for calibration failures
import json
import re
from pathlib import Path
from collections import Counter

# Failure counters
failed_sites = Counter()
failed_sensors = Counter()
failed_combos = Counter()

# Total sample counters
total_sites = Counter()
total_sensors = Counter()
total_combos = Counter()

total_samples = 0
total_calib_failures = 0

for json_path in Path("../data/sensori-data").rglob("info.json"):
    data = json.loads(json_path.read_text())
    total_samples += 1

    filename = data.get("Filename", "")
    site_match = re.search(r"MS\d+", filename)
    sensor_match = re.search(r"_(MM|AX6)_", filename)

    site = site_match.group(0) if site_match else "Unknown_Site"
    sensor = sensor_match.group(1) if sensor_match else "Unknown_Sensor"
    combo = f"{site} - {sensor}"

    # Track overall totals
    total_sites[site] += 1
    total_sensors[sensor] += 1
    total_combos[combo] += 1

    # Check if calibration failed
    is_calib_ok = data.get("eligibility", {}).get("calibration_ok", True) and data.get("CalibOK", 1) == 1
    if not is_calib_ok:
        total_calib_failures += 1
        failed_sites[site] += 1
        failed_sensors[sensor] += 1
        failed_combos[combo] += 1

print(f"Total Samples: {total_samples} | Total Calibration Failures: {total_calib_failures} ({total_calib_failures/total_samples*100:.1f}%)\n")

print("Failures by Site (Failure Rate per Site):")
for site, total in total_sites.items():
    fails = failed_sites[site]
    pct = (fails / total) * 100
    print(f"  - {site}: {fails}/{total} ({pct:.1f}%)")

print("\nFailures by Sensor (Failure Rate per Sensor):")
for sensor, total in total_sensors.items():
    fails = failed_sensors[sensor]
    pct = (fails / total) * 100
    print(f"  - {sensor}: {fails}/{total} ({pct:.1f}%)")

print("\nFailures by Site & Sensor Combination:")
for combo, total in total_combos.items():
    fails = failed_combos[combo]
    pct = (fails / total) * 100
    print(f"  - {combo}: {fails}/{total} ({pct:.1f}%)")

Total Samples: 2403 | Total Calibration Failures: 1314 (54.7%)

Failures by Site (Failure Rate per Site):
  - MS25: 590/898 (65.7%)
  - MS10: 94/187 (50.3%)
  - MS24: 600/1271 (47.2%)
  - MS21: 30/47 (63.8%)

Failures by Sensor (Failure Rate per Sensor):
  - AX6: 1234/1996 (61.8%)
  - MM: 80/407 (19.7%)

Failures by Site & Sensor Combination:
  - MS25 - AX6: 590/898 (65.7%)
  - MS10 - AX6: 87/142 (61.3%)
  - MS24 - AX6: 527/909 (58.0%)
  - MS24 - MM: 73/362 (20.2%)
  - MS10 - MM: 7/45 (15.6%)
  - MS21 - AX6: 30/47 (63.8%)


In [6]:
import json
from pathlib import Path
import pandas as pd

rows = []

for json_path in Path("../data/sensori-data").rglob("info.json"):
    data = json.loads(json_path.read_text())

    # Extract calibration diagnostics
    rows.append({
        "sample_id": json_path.parent.name,
        "calib_ok": data.get("CalibOK"),
        "error_before_mg": data.get("CalibErrorBefore(mg)"),
        "error_after_mg": data.get("CalibErrorAfter(mg)"),
        "num_samples_used": data.get("CalibNumSamples"),
        "num_iters": data.get("CalibNumIters"),
        "x_slope": data.get("CalibxSlope"),
        "y_slope": data.get("CalibySlope"),
        "z_slope": data.get("CalibzSlope"),
        "x_intercept": data.get("CalibxIntercept"),
        "y_intercept": data.get("CalibyIntercept"),
        "z_intercept": data.get("CalibzIntercept"),
    })

df = pd.DataFrame(rows)

# Split into passed and failed calibration
passed = df[df["calib_ok"] == 1]
failed = df[df["calib_ok"] == 0]

print("=== CALIBRATION FAILURE DIAGNOSTICS ===")
print(f"Total Failed: {len(failed)}")
print("\nMean Stationary Samples Used:")
print(f"  - Passed: {passed['num_samples_used'].mean():.1f}")
print(f"  - Failed: {failed['num_samples_used'].mean():.1f}")

print("\nMean Calibration Error After (mg):")
print(f"  - Passed: {passed['error_after_mg'].mean():.2f} mg")
print(f"  - Failed: {failed['error_after_mg'].mean():.2f} mg")

print("\nOptimizer Iterations:")
print(f"  - Passed: {passed['num_iters'].mean():.1f}")
print(f"  - Failed: {failed['num_iters'].mean():.1f}")

=== CALIBRATION FAILURE DIAGNOSTICS ===
Total Failed: 1314

Mean Stationary Samples Used:
  - Passed: 39274.6
  - Failed: 39056.1

Mean Calibration Error After (mg):
  - Passed: 2.55 mg
  - Failed: 36.06 mg

Optimizer Iterations:
  - Passed: 218.0
  - Failed: 7.9
